In [1]:
# @title
# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip uninstall -y datasets huggingface_hub fsspec

!pip install -q \
datasets==2.19.2 \
huggingface_hub==0.23.5 \
fsspec==2024.3.1 \
pyarrow \
fastparquet \
pandas \
tqdm \
xxhash

print("Installation complete.")
print("Please restart the runtime before continuing.")

import datasets
import huggingface_hub
import fsspec

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("fsspec:", fsspec.__version__)


# ============================================================
# STEP 2: IMPORTS
# Run after runtime restart
# ============================================================

import os
import re
import logging
import pandas as pd
import xxhash

from tqdm.auto import tqdm
from datasets import load_dataset, Dataset

logging.basicConfig(
  level=logging.INFO,
  format="%(asctime)s - %(levelname)s - %(message)s"
)

print("Installation complete.")
print("Please Runtime -> Restart Session before continuing.")

Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: huggingface_hub 1.19.0
Uninstalling huggingface_hub-1.19.0:
  Successfully uninstalled huggingface_hub-1.19.0
Found existing installation: fsspec 2025.3.0
Uninstalling fsspec-2025.3.0:
  Successfully uninstalled fsspec-2025.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.8/402.8 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 62.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.38.0 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.23.5 which is incompatible.
gcsfs 2025.3.0 requ

In [5]:
# ============================================================
# CONFIG
# ============================================================

OUTPUT_DIR = "/content/unified_corpus"

MIN_CODE_LEN = 20
MAX_CODE_LEN = 500 #We can increase the count here, if we want more number of datasets.
MIN_DESC_LEN = 15

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================

def normalize_text(text):

  if text is None:
    return ""
  return str(text).strip()

def normalize_code(code):
  return re.sub(r"\s+", " ", str(code)).strip()

def code_hash(code):
  return xxhash.xxh64(
  normalize_code(code).encode("utf-8")).hexdigest()

def valid_record(code, desc):

  if not code:
    return False

  if not desc:
    return False

  if len(code) < MIN_CODE_LEN:
    return False

  if len(code) > MAX_CODE_LEN:
    return False

  if len(desc) < MIN_DESC_LEN:
    return False

  if len(desc.split()) < 3:
    return False

  return True

In [2]:
# ============================================================
# DATASET SIZE INSPECTION
# ============================================================

languages = [
"python",
"java"
#"javascript",
#"php",
#"ruby",
#"go"
]

print("=" * 80)
print("VERIFY CODESEARCHNET ACCESS")
print("=" * 80)

stats = []
total_rows = 0

for lang in languages:

  try:

    ds = load_dataset(
        "code_search_net",
        lang,
        split="train"
    )

    rows = len(ds)

    total_rows += rows

    print(
        f"{lang:<12} {rows:,}"
    )

    stats.append({
        "dataset": "CodeSearchNet",
        "language": lang,
        "rows": rows
    })

  except Exception as e:

    print(
        f"{lang:<12} FAILED"
    )

    print(e)

  stats_df = pd.DataFrame(stats)

  print("\nSummary")
  print(stats_df)

  print(
  f"\nTotal Rows: {total_rows:,}"
)

VERIFY CODESEARCHNET ACCESS


Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

python       412,178

Summary
         dataset language    rows
0  CodeSearchNet   python  412178

Total Rows: 412,178


Generating train split:   0%|          | 0/454451 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26909 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/15328 [00:00<?, ? examples/s]

java         454,451

Summary
         dataset language    rows
0  CodeSearchNet   python  412178
1  CodeSearchNet     java  454451

Total Rows: 866,629


In [3]:
# ============================================================
# OPTIONAL:
# VERIFY CODEXGLUE CODE-TO-TEXT DATASET
# ============================================================

print("\n" + "=" * 80)
print("VERIFY CODEXGLUE")
print("=" * 80)

codexglue_langs = [
"python",
"java"
#"javascript",
#"php",
#"ruby",
#"go"
]

for lang in codexglue_langs:

  try:

    ds = load_dataset(
        "code_x_glue_ct_code_to_text",
        lang,
        split="train"
    )

    print(
        f"{lang:<12} {len(ds):,}"
    )

  except Exception as e:

    print(
        f"{lang:<12} FAILED"
    )

    print(e)


VERIFY CODEXGLUE


Generating train split:   0%|          | 0/251820 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13914 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/14918 [00:00<?, ? examples/s]

python       251,820


Generating train split:   0%|          | 0/164923 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5183 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10955 [00:00<?, ? examples/s]

java         164,923


In [6]:

# ============================================================
# LOAD CODESEARCHNET
# ============================================================

records = []

print("\n" + "=" * 80)
print("LOADING CODESEARCHNET")
print("=" * 80)

for lang in languages:

  try:

    ds = load_dataset(
        "code_search_net",
        lang,
        split="train"
    )

    print(
        f"\nLoading {lang}"
    )

    print(
        "Columns:",
        ds.column_names
    )

    for row in tqdm(
        ds,
        desc=f"CSN-{lang}"
    ):

        code = normalize_text(
            row.get(
                "whole_func_string"
            )
        )

        desc = normalize_text(
            row.get(
                "func_documentation_string"
            )
        )

        if not valid_record(
            code,
            desc
        ):
            continue

        records.append({
            "language": lang,
            "code": code,
            "description": desc,
            "source": "codesearchnet"
        })

  except Exception as e:

    print(
        f"Failed loading {lang}"
    )

    print(e)

print(
f"\nCodeSearchNet records collected: {len(records):,}"
)

# ============================================================
# LOAD CODEXGLUE
# ============================================================

print("\n" + "=" * 80)
print("LOADING CODEXGLUE")
print("=" * 80)

for lang in codexglue_langs:

  try:

    ds = load_dataset(
        "code_x_glue_ct_code_to_text",
        lang,
        split="train"
    )

    before = len(records)

    for row in tqdm(
        ds,
        desc=f"CodeXGLUE-{lang}"
    ):

        code = normalize_text(
            row.get("code")
        )

        desc = normalize_text(
            row.get("docstring")
        )

        if not valid_record(
            code,
            desc
        ):
            continue

        records.append({
            "language": lang,
            "code": code,
            "description": desc,
            "source": "codexglue"
        })

    print(
        f"{lang}: added {len(records)-before:,}"
    )

  except Exception as e:

    print(
        f"CodeXGLUE {lang} failed"
    )

    print(e)

# ============================================================
# SAFETY CHECK
# ============================================================

if len(records) == 0:

  raise RuntimeError(
    "No records loaded. Verify dataset access."
)

print(f"total number pf records: ", len(records));

# ============================================================

# DATAFRAME

# ============================================================

df = pd.DataFrame(records)

if df.empty:

  raise RuntimeError(
    "DataFrame is empty."
)

print(
f"\nRows before dedup: {len(df):,}"
)

# ============================================================
# DEDUPLICATION
# ============================================================

df["hash"] = df["code"].apply(
code_hash
)

before = len(df)

df = (
df
.drop_duplicates(
subset=["hash"]
)
.drop(
columns=["hash"]
)
.reset_index(drop=True)
)

print(
f"Duplicates removed: {before-len(df):,}"
)

print(
f"Rows after dedup: {len(df):,}"
)

# ============================================================
# SHUFFLE
# ============================================================

df = df.sample(
frac=1,
random_state=42
).reset_index(drop=True)

# ============================================================

# STATS

# ============================================================

print("\nSource Distribution")
print(
df["source"].value_counts()
)

print("\nLanguage Distribution")
print(
df["language"].value_counts()
)

# ============================================================

# SAVE PARQUET

# ============================================================

PARQUET_FILE = (
f"{OUTPUT_DIR}/unified_corpus.parquet"
)

df.to_parquet(
PARQUET_FILE,
index=False
)

print(
f"\nSaved parquet: {PARQUET_FILE}"
)

# ============================================================

# HF DATASET

# ============================================================

hf_ds = Dataset.from_pandas(
df,
preserve_index=False
)

split_1 = hf_ds.train_test_split(
test_size=0.10,
seed=42
)

train_ds = split_1["train"]
temp_ds = split_1["test"]

split_2 = temp_ds.train_test_split(
test_size=0.50,
seed=42
)

valid_ds = split_2["train"]
test_ds = split_2["test"]

print("\nDataset Sizes")

print(
"Train:",
len(train_ds)
)

print(
"Validation:",
len(valid_ds)
)

print(
"Test:",
len(test_ds)
)

train_ds.save_to_disk(
f"{OUTPUT_DIR}/train"
)

valid_ds.save_to_disk(
f"{OUTPUT_DIR}/validation"
)

test_ds.save_to_disk(
f"{OUTPUT_DIR}/test"
)

print("\nCompleted Successfully")

print("\nExample Record")
print(train_ds[0])



LOADING CODESEARCHNET

Loading python
Columns: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url']


CSN-python:   0%|          | 0/412178 [00:00<?, ?it/s]


Loading java
Columns: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url']


CSN-java:   0%|          | 0/454451 [00:00<?, ?it/s]


CodeSearchNet records collected: 404,369

LOADING CODEXGLUE


...

java: added 99,682
total number pf records:  601079

Rows before dedup: 601,079
Duplicates removed: 196,710
Rows after dedup: 404,369

Source Distribution
source
codesearchnet    404369
Name: count, dtype: int64

Language Distribution
language
java      261177
python    143192
Name: count, dtype: int64

Saved parquet: /content/unified_corpus/unified_corpus.parquet

Dataset Sizes
Train: 363932
Validation: 20218
Test: 20219


Saving the dataset (0/1 shards):   0%|          | 0/363932 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20218 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20219 [00:00<?, ? examples/s]


Completed Successfully

Example Record
{'language': 'python', 'code': 'def from_EV(E, V):\n        """\n        Creates an instance of a Gamma Prior  by specifying the Expected value(s)\n        and Variance(s) of the distribution.\n\n        :param E: expected value\n        :param V: variance\n        """\n        a = np.square(E) / V\n        b = E / V\n        return Gamma(a, b)', 'description': 'Creates an instance of a Gamma Prior  by specifying the Expected value(s)\n        and Variance(s) of the distribution.\n\n        :param E: expected value\n        :param V: variance', 'source': 'codesearchnet'}
